# Decision Trees and Random Forest – Loan Risk Modeling

**🔑 TARGET & FEATURE CLARITY**

Loan_Status   →  Approved / Rejected / Pending (or similar)


# 🌳 Decision Trees, Random Forest & XGBoost
## Loan Status Prediction using HDFC Loan Dataset

### Objective
- Build tree-based classification models
- Predict loan approval status
- Compare Decision Tree, Random Forest, and XGBoost
- Understand feature importance for business decisions


## 📘 Tree-Based Models Overview

### Decision Tree
- Rule-based splits
- Easy to interpret
- Can overfit

### Random Forest
- Ensemble of decision trees
- Reduces variance
- More stable predictions



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/hdfc_loan_dataset_full_enriched.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Loan_ID,Bank,Customer_Name,Gender,Married,Dependents,Education,Employment_Status,Applicant_Income,Coapplicant_Income,...,Loan_to_Annual_Income,Customer_Sentiment,Religion,State,City,PIN_Code,Aadhaar_Synthetic,Phone_Number,Email,Occupation
0,HDFC100001,HDFC Bank,Rohan Verma,Male,No,2,Graduate,Salaried,56976,0,...,11.747,Positive,Hindu,Delhi,Dwarka,857743,6.940000e+11,9068671773,rohan.verma@example.in,Farmer
1,HDFC100002,HDFC Bank,Rohan Verma,Male,No,0,Graduate,Unemployed,1856,0,...,40.381,Negative,Hindu,Punjab,Ludhiana,863836,6.330000e+11,9990772625,rohan.verma@example.in,Civil Engineer
2,HDFC100003,HDFC Bank,Aditya Nair,Female,Yes,0,Graduate,Salaried,64553,0,...,3.082,Positive,Hindu,Maharashtra,Nagpur,834796,1.660000e+11,9195085016,aditya.nair@example.in,Medical Representative
3,HDFC100004,HDFC Bank,Ananya Joshi,Male,No,0,Graduate,Salaried,88450,0,...,0.621,Negative,Hindu,Gujarat,Vadodara,438590,5.528183e+10,9179335548,ananya.joshi@example.in,Marketing Executive
4,HDFC100005,HDFC Bank,Harpreet Singh,Male,Yes,3,Graduate,Self-Employed,9539,0,...,1.736,Neutral,Sikh,West Bengal,Kolkata,495224,1.560000e+11,9795137116,harpreet.singh@example.in,Shopkeeper


## 📊 Dataset Overview

The dataset contains:
- Applicant demographics
- Financial indicators
- Credit behavior
- Geographic and institutional data
- Textual fields (excluded from modeling)

Target variable:
➡️ `Loan_Status`


In [ ]:
drop_cols = [
    'Loan_ID', 'Customer_Name', 'Phone_Number', 'Email',
    'Aadhaar_Synthetic', 'Application_Text',
    'Customer_Feedback', 'Agent_Notes', 'Unnamed: 39', 'Bank'
]

df_model = df.drop(columns=drop_cols, errors = "ignore")

In [ ]:
# Fill missing categorical values
df_model['Business_Type'].fillna('Unknown', inplace=True)
df_model['Co-signer_Relationship'].fillna('None', inplace=True)

df_model.isnull().sum()

/tmp/ipykernel_1026/761070405.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model['Business_Type'].fillna('Unknown', inplace=True)
/tmp/ipykernel_1026/761070405.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

,0
Gender,0
Married,0
Dependents,0
Education,0
Employment_Status,0
Applicant_Income,0
Coapplicant_Income,0
Loan_Amount,0
Loan_Term_Months,0
Credit_History,0


In [ ]:
target = 'Loan_Status'

X = df_model.drop(columns=[target])
y = df_model[target]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'
)

In [ ]:
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_features, cat_features

(['Dependents',
  'Applicant_Income',
  'Coapplicant_Income',
  'Loan_Amount',
  'Loan_Term_Months',
  'Credit_History',
  'Age',
  'CIBIL_Score',
  'Annual_Household_Income',
  'Debt_to_Income_Ratio',
  'Existing_EMIs',
  'Number_of_Previous_Loans',
  'Default_History_Count',
  'Employment_Length_Years',
  'Asset_Value',
  'Monthly_Expense',
  'Loan_to_Annual_Income',
  'PIN_Code'],
 ['Gender',
  'Married',
  'Education',
  'Employment_Status',
  'Property_Area',
  'Purpose_of_Loan',
  'Business_Type',
  'Guarantor',
  'Co-signer_Relationship',
  'Organization_Type',
  'Region_Branch',
  'Mobile_Verified',
  'Email_Verified',
  'Institutional_Relationships',
  'Customer_Sentiment',
  'Religion',
  'State',
  'City',
  'Occupation'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **🌳 DECISION TREE**

In [ ]:
dt_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=6, random_state=42))
])

dt_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Married',
                                                   'Education',
                                                   'Employment_Status',
                                                   'Property_Area',
                                                   'Purpose_of_Loan',
                                                   'Business_Type', 'Guarantor',
                                                   'Co-signer_Relationship',
                                                   'Organization_Type',
                                                   'Region_Branch',
                                                   'Mobile_Verified',
                                                   'Email_Verified',
                                                   'Institutional_Relationships',
                                                   'Customer_Sentiment',
                                                   'Religion', 'State', 'City',
                                                   'Occupation'])])),
                ('model',
                 DecisionTreeClassifier(max_depth=6, random_state=42))])

In [ ]:
dt_pred = dt_pipeline.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.83
              precision    recall  f1-score   support

    Approved       0.88      0.85      0.87       131
    Rejected       0.74      0.78      0.76        69

    accuracy                           0.83       200
   macro avg       0.81      0.82      0.81       200
weighted avg       0.83      0.83      0.83       200



# **🌲 RANDOM FOREST**

In [ ]:
rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Married',
                                                   'Education',
                                                   'Employment_Status',
                                                   'Property_Area',
                                                   'Purpose_of_Loan',
                                                   'Business_Type', 'Guarantor',
                                                   'Co-signer_Relationship',
                                                   'Organization_Type',
                                                   'Region_Branch',
                                                   'Mobile_Verified',
                                                   'Email_Verified',
                                                   'Institutional_Relationships',
                                                   'Customer_Sentiment',
                                                   'Religion', 'State', 'City',
                                                   'Occupation'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, n_estimators=300,
                                        random_state=42))])

In [ ]:
rf_pred = rf_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.85
              precision    recall  f1-score   support

    Approved       0.85      0.93      0.89       131
    Rejected       0.84      0.70      0.76        69

    accuracy                           0.85       200
   macro avg       0.85      0.81      0.83       200
weighted avg       0.85      0.85      0.85       200



# **📊 MODEL COMPARISON**

In [ ]:
pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

,Model,Accuracy
0,Decision Tree,0.83
1,Random Forest,0.85


# **UNSOLVED EXERCISE**

🧠 Exercise 1 (Unsolved)
📌 Feature Importance Analysis
Task

Extract feature importance from:

1. Random Forest

2. Decision Tree

Identify the top 5 most important features.

Answer:

Which features influence loan approval the most?

Do both models agree on the top features?

In [ ]:
features = rf_pipeline.named_steps["preprocess"].get_feature_names_out()

rf_importance = rf_pipeline.named_steps["model"].feature_importances_
df = pd.DataFrame({
    'feature' : features,
    'Importance' : rf_importance
}).sort_values(by='Importance', ascending=False)
print(df.head(5))

                                feature  Importance
131     remainder__Debt_to_Income_Ratio    0.143680
138    remainder__Loan_to_Annual_Income    0.086843
130  remainder__Annual_Household_Income    0.071763
132            remainder__Existing_EMIs    0.064575
123         remainder__Applicant_Income    0.059823


In [ ]:
dt_importance = dt_pipeline.named_steps['model'].feature_importances_

dt_feat = pd.DataFrame({
    'Feature': features,
    'Importance': dt_importance
}).sort_values(by='Importance', ascending=False)

print("Top 5 Decision Tree Features:")
print(dt_feat.head(5))

Top 5 Decision Tree Features:
                              Feature  Importance
131   remainder__Debt_to_Income_Ratio    0.504795
138  remainder__Loan_to_Annual_Income    0.192869
127         remainder__Credit_History    0.126461
134  remainder__Default_History_Count    0.027399
129            remainder__CIBIL_Score    0.023625


🧠 Exercise 2 (Unsolved)
📌 Model Tuning & Overfitting Check
Task

1. Train a Decision Tree with:

max_depth=None

2. Train another Decision Tree with:

max_depth=4

3. Compare:

Training accuracy

Test accuracy

4. Answer:

Which model overfits?

Why does depth control matter in loan risk models?

In [ ]:
dt_pipeline_full = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=None, random_state=42))
])

dt_pipeline_full.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Married',
                                                   'Education',
                                                   'Employment_Status',
                                                   'Property_Area',
                                                   'Purpose_of_Loan',
                                                   'Business_Type', 'Guarantor',
                                                   'Co-signer_Relationship',
                                                   'Organization_Type',
                                                   'Region_Branch',
                                                   'Mobile_Verified',
                                                   'Email_Verified',
                                                   'Institutional_Relationships',
                                                   'Customer_Sentiment',
                                                   'Religion', 'State', 'City',
                                                   'Occupation'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

In [ ]:
dt_pipeline_small = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])

dt_pipeline_small.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'Married',
                                                   'Education',
                                                   'Employment_Status',
                                                   'Property_Area',
                                                   'Purpose_of_Loan',
                                                   'Business_Type', 'Guarantor',
                                                   'Co-signer_Relationship',
                                                   'Organization_Type',
                                                   'Region_Branch',
                                                   'Mobile_Verified',
                                                   'Email_Verified',
                                                   'Institutional_Relationships',
                                                   'Customer_Sentiment',
                                                   'Religion', 'State', 'City',
                                                   'Occupation'])])),
                ('model',
                 DecisionTreeClassifier(max_depth=4, random_state=42))])

In [ ]:
print("Full depth training accuracy:", dt_pipeline_full.score(X_train, y_train))
print("Full depth testing accuracy:", dt_pipeline_full.score(X_test, y_test))

print("Small depth training accuracy:", dt_pipeline_small.score(X_train, y_train))
print("Small depth testing accuracy:", dt_pipeline_small.score(X_test, y_test))

Full depth training accuracy: 1.0
Full depth testing accuracy: 0.805
Small depth training accuracy: 0.91
Small depth testing accuracy: 0.82
